# STEP 10 — 확정 재학습 + holdout (한 세션에서 끝까지)

03(학습)과 05(평가)를 **한 노트북에 이어붙인 것**입니다.

## ★ 붙일 게 크롭 두 개뿐입니다

이 프로젝트에서 시간을 제일 많이 날린 곳이 **노트북 사이 인계**였습니다.
세션이 달라 체크포인트가 안 넘어가고, 넘어가도 JSON 이 빠지고, 낡은 버전을
붙이고 … 한 세션에서 끝내면 그 문제가 통째로 사라집니다.

| Kaggle 입력 | |
|---|---|
| `dogskin-f320` | 1단계 (STEP 9-A 채택) |
| `dogskin-m25` | 2단계 |

**이전 release 는 안 붙입니다.** 두 단계 다 처음부터 학습하고, 그대로 holdout 까지
갑니다. 중간에 넘길 게 없습니다.

## 🚨 이 노트북은 holdout 을 엽니다

**holdout 은 학습에도 모델 선택에도 한 번도 안 쓴 마지막 시험지입니다.**
열어보고 설정을 바꾸면 그 순간 holdout 도 val 이 됩니다.

| | 노트북 |
|---|---|
| 설정을 **고르는** 실험 | 03b / 03c / 03d / 03e / 04 — holdout 안 엶 |
| 고른 설정을 **확정 측정** | **이 노트북** — holdout 엶 |

> 이미 고른 설정을 확인할 때만 쓰세요. 여기서 나온 숫자를 보고 설정을
> 다시 고르면 안 됩니다. 그게 STEP 5 의 실패였습니다.

## 이번 실행에서 재는 것

STEP 9-A 에서 1단계 입력을 `f320` 으로 정했습니다 (val AUROC 0.8272 → 0.9477).
그게 **처음 보는 개체**에서도 유지되는지 확인합니다.

| 확인할 것 | STEP 8 | 목표 |
|---|---:|---:|
| 1단계 holdout AUROC | 0.7666 | ≥0.80 |
| 헛알림 | 59.2% | 줄어야 함 |
| 최종 7종 macro-F1 | 0.4594 | 올라야 함 |

## ⚠️ 2단계도 다시 학습합니다 — 읽을 때 감안할 것

2단계(m2.5 · resnet50)는 설정이 그대로인데도 새로 학습합니다. 인계를 없애는
대가입니다. 그래서 최종 숫자의 변화에는 **1단계 효과 + 2단계 재학습 잡음**이
섞입니다 (2단계 macro-F1 의 실행 간 잡음은 ±0.02 쯤).

읽는 법: **1단계 holdout AUROC 는 순수하게 1단계 것**이므로 그걸 1순위로 보고,
최종 7종 macro-F1 은 ±0.02 를 감안해서 읽으세요.

⚠️ 2단계 백본은 아직 미확정입니다 (STEP 9 판정 보류 — 기준선 미수렴).
여기서 resnet50 을 확정하는 게 아니라, 채택된 기준선을 그대로 쓰는 것입니다.

## 예상 시간

1단계 18에폭 + 2단계 25에폭 + 교란 검사 + holdout 평가 ≈ **4시간**.
아래 시간 추정 셀이 실측으로 다시 알려줍니다.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or "main"
if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

# ⚠️ Colab/Kaggle 에는 numpy·pandas·sklearn 이 이미 있지만 **임대 GPU 이미지엔
#    torch 만 있는 경우가 많습니다** (런팟에서 `No module named 'pandas'` 로
#    막혔습니다). 그렇다고 매번 다 깔면 Colab 에서 버전이 흔들리므로
#    **없는 것만** 깝니다.
_NEED = {                       # import 이름 → pip 이름
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "PIL": "Pillow",
    "sklearn": "scikit-learn", "cv2": "opencv-python-headless", "tqdm": "tqdm",
    "matplotlib": "matplotlib", "timm": "timm", "imagehash": "imagehash",
    "pytorch_grad_cam": "grad-cam", "albumentations": "albumentations",
}
import importlib.util as _ilu

_PKGS = [pip for mod, pip in _NEED.items() if _ilu.find_spec(mod) is None]
if _PKGS:
    print(f"[env] 없는 패키지 {len(_PKGS)}개를 깝니다: {_PKGS}")
else:
    print("[env] 필요한 패키지가 전부 있습니다 — 설치를 건너뜁니다")

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = not _PKGS          # 깔 게 없으면 이미 성공입니다
if _PKGS and _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-08-25.1"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


## 0-b. 로컬에서 만든 데이터 불러오기 + 중단 대비

한국 PC 에서 `prepare_local.py` 로 전처리한 `dogskin_prepared.zip` 을 가져옵니다.

> 🚨 **AI Hub 는 해외 IP 다운로드를 차단**해서 Colab 에서는 원본을 받을 수 없습니다.
> 다운로드·전처리는 로컬에서, 학습만 여기서 합니다.
> → [`docs/cautions/06`](../docs/cautions/06_해외IP_다운로드_차단_우회.md)

**준비**
- **Colab** : `dogskin_prepared.zip` 을 Google Drive 에 올려두세요
- **Kaggle** : 같은 zip 을 Dataset 으로 올리고 우측 **Add Input** 으로 붙이세요.
  Kaggle 이 zip 을 알아서 풀어두므로 `crops/` `manifests/` 가 바로 보입니다.
  → 자세한 절차: [`docs/cautions/09`](../docs/cautions/09_세션이_끊겼을_때.md)

### ⚠️ Colab 세션은 예고 없이 끊깁니다

3번+4번 학습이 합쳐서 1시간 반쯤 걸립니다. 그 사이에 세션이 끊기면
**`/content` 가 통째로 사라집니다** — 크롭 이미지도, 체크포인트도요.

그래서 체크포인트는 매 에폭 **Drive 로 복사**해 둡니다.
끊기면 이 노트북을 **위에서부터 그냥 다시 돌리세요.** 끝난 학습은 건너뛰고,
끊긴 학습은 그 다음 에폭부터 이어갑니다 (옵티마이저 상태까지 복원).

| 상황 | 다시 돌렸을 때 |
|---|---|
| 3번은 끝, 4번 도중에 끊김 | 3번 `⏭️ 건너뜁니다` → 4번 `▶️ epoch N 부터 이어서` |
| 4번을 20에폭에서 끊김 | 21에폭부터 (앞의 20에폭 다시 안 함) |
| 다 끝났는데 또 돌림 | 둘 다 건너뜀, 평가만 다시 |
| 에폭을 더 늘리고 싶다 | `epochs` 만 키우면 이어서 연장합니다 |
| 처음부터 다시 하고 싶다 | `train.fit(..., resume=False)` |

> 💡 **Drive 마운트를 건너뛰면 이 보호가 없습니다.** 아래 셀이 경고합니다.
> 체크포인트는 실험당 약 600MB 를 씁니다 (`best.pt` 200MB + `last.pt` 400MB).
> Drive 용량이 빡빡하면 Drive 의 `MyDrive/dogskin_work/checkpoints/*/last.pt` 를 지우세요
> (`best.pt` 만 남기면 이어받기는 못 하지만 평가는 됩니다).


In [ ]:
# Drive 마운트는 **진짜 Colab VM** 에서만 시도합니다.
# ⚠️ Kaggle 에도 google.colab 패키지와 /content 가 있어서, 환경 판정을 잘못하면
#    Kaggle 에서 drive.mount() 를 부르고 NotImplementedError 로 죽습니다.
if env.can_mount_drive():
    env.mount_drive()
else:
    print(f"[env] {E.env} — Drive 마운트 없이 진행합니다")

# 전처리 결과를 붙입니다. 두 가지 형태를 다 받습니다:
#   · Colab  : Drive 의 dogskin_prepared.zip → 로컬 디스크로 해제
#   · Kaggle : /kaggle/input/<데이터셋>/crops,manifests → 링크만 연결
#              (Kaggle 은 업로드한 zip 을 알아서 풀어둡니다. 복사하면 20GB 제한에 걸려요)
env.load_prepared()          # 경로를 직접 주려면: env.load_prepared("/kaggle/input/dogskin-prepared")

# ⚠️ train 은 아래 셀에서 import 하지만 여기서 먼저 씁니다 — 여기서 불러둡니다.
#    (실제로 NameError 로 5분 돌다 죽었습니다. 셀 순서를 믿지 말 것)
from src import train

# ── 예전 실행을 붙였으면 가져옵니다 (이 노트북에서는 **안 붙여도 됩니다**) ──
#    이 노트북은 두 단계를 다 학습하고 그대로 holdout 까지 가므로 인계가
#    필요 없습니다. 붙어 있으면 그 단계를 건너뛰어 시간을 아끼고,
#    없으면 조용히 처음부터 학습합니다.
_prev = train.import_previous_run(verbose=True)
if _prev and _prev.get("checkpoints"):
    print(f"\n[인계] 가져온 실험: {_prev['checkpoints']}")
    print("   설정이 그대로인 단계는 아래 학습 셀에서 ⏭️ 로 건너뜁니다.")
    print("   ⚠️ 다시 학습하고 싶으면 train.fit(..., resume=False) 로 부르세요.")
else:
    print("\n[인계] 붙어 있는 이전 실행이 없습니다 — 두 단계 다 처음부터 학습합니다.")

# 세션이 끊겨도 남는 저장소 확인
_persist = env.persist_root()
if _persist is None:
    print("\n🚨 세션 밖 저장소가 없습니다 — 지금 학습하면 끊길 때 체크포인트가 사라집니다.")
    print("   위 셀에서 Drive 마운트가 됐는지 확인하세요 (env.mount_drive()).")
else:
    print(f"\n✅ 중단 대비 저장소: {_persist}")
    if E.env == "kaggle":
        print("   ⚠️ Kaggle 은 세션이 끝나면 /kaggle/working 이 사라질 수 있습니다.")
        print("      · 짧게 확인만 할 때  : 그냥 진행 (세션 안에서는 이어받기가 됩니다)")
        print("      · 긴 학습을 돌릴 때  : 우측 상단 [Save Version] →")
        print("                             'Save & Run All (Commit)' 로 돌리세요.")
        print("                             브라우저를 닫아도 끝까지 돌고, 출력이 보존됩니다.")
        print("      · 설정에 Persistence 항목이 보이면 'Files' 로 켜두면 더 안전합니다")
    else:
        print("   매 에폭 체크포인트를 여기로 복사합니다. 세션이 끊기면 노트북을 처음부터")
        print("   다시 돌리세요 — 끝난 학습은 건너뛰고 끊긴 학습만 이어서 합니다.")

In [ ]:
import torch
from src import labels, split, crop, data, models, train, evaluate, stages
from src.config import CLASSES_STAGE1, NORMAL_LABEL

# ★ GPU 없이 진행하면 20~30배 느립니다. 없으면 여기서 멈춥니다.
#   (Colab 무료 한도를 넘기면 말없이 CPU 런타임을 줍니다 — 이걸 막습니다)
env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"

df = labels.load(env.work_root()/"manifests"/"manifest_final.parquet")
print(f"{len(df):,}행 / 개체 {df['animal_id'].nunique():,}마리")
print("클래스 분포:", df["label"].value_counts().to_dict())
print("사용 가능한 크롭 태그:", crop.available_tags())

# 로컬에서 개체 단위 분할까지 끝냈으므로 fold/holdout 컬럼이 들어 있습니다
split.verify(df, fold=0, strict=True)

## 1. 크롭 검증 — 숫자로 먼저, 눈으로는 딱 하나만

여기서 확인해야 하는 건 "병변이 맞는가" 가 **아닙니다.** 그건 수의사의 일이고,
저 데이터의 라벨은 이미 수의사가 붙인 것입니다. 우리가 확인할 건 다른 겁니다:

> **크롭이 라벨을 다른 경로로 흘리고 있지 않은가.**

무슨 뜻이냐면 — A7(정상)에도 라벨러가 "촬영한 피부 부위" 박스를 찍어놨습니다.
그래서 A7 도 `area_ratio` 값이 있습니다. 그 자체는 정상인데, 만약
**정상 박스가 병변 박스보다 계통적으로 크다면** 크롭의 확대 배율만 봐도
정상/병변이 티가 납니다. 그러면 모델은 피부를 안 보고 **줌 레벨을 셉니다.**

이런 지름길(shortcut)은 검증 점수를 **높게** 만듭니다. 그래서 숫자로 잡아야 합니다.
`crop.audit()` 이 그걸 포함해 5가지를 재줍니다 — 의학 지식이 필요 없습니다.

In [ ]:
# ★ m2.5 확정 (2026-08-22, STEP 4C). 03c 에서 같은 실행으로 비교한 결과
#   배율 하락 23.2% → 18.8%, A1 recall 0.560 → 0.622.
#   근거: docs/results/STEP4C_크롭비교_실측.md
#
#   ⚠️ **1단계는 이 값을 따라오지 않습니다.** 아래 18번 셀의
#   crop.choose_stage1_tag() 가 따로 고릅니다 (우선순위 f320 > full > ROI).
#   ROI 크롭은 창 크기가 정답을 흘려서 1단계에 못 씁니다.
BEST_CROP = "m2.5"                       # 2단계 크롭. 병변 형태를 보려면 ROI 가 필요합니다
# ★ 384 채택 (2026-08-21 해상도 실험). 감사 [2] 와 맞춰야 의미가 있습니다.
#   224 → 384 로 올려 배율 하락이 1단계 17.2%→9.1%, 2단계 29.5%→20.4% 로 줄었습니다.
#   디스크 크롭이 512px 라 재크롭 없이 가능하고, 배치는 자동으로 줄어듭니다(T4 → 12).
#   근거: docs/results/STEP4A_베이스라인_실측.md
IMG_SIZE  = 384

# ★ STEP 6 (03d) 에서 1단계 설정을 갈아끼웠습니다 — 2×2 실측 결과입니다.
#   근거: docs/results/STEP6_1단계_2x2_실측.md
#
#     resnet50   / default      AUROC 0.8116   흐림 하락 55.4%   ← 이전
#     effnetv2_s / photometric  AUROC 0.8284   흐림 하락 14.0%   ← 채택
#
#   ⚠️ val AUROC 최고는 effnetv2_s/default(0.8359) 였지만 **안 골랐습니다.**
#      흐림 하락 47.6% 로 화질 지름길에 기대고 있고, AUROC 차이 0.0075 는
#      잡음(±0.01) 안입니다. val 점수로 고르다가 holdout 에서 무너진 게
#      STEP 5 의 실패였습니다 (0.8143 → 0.7412).
STAGE1_MODEL = "effnetv2_s"
STAGE1_AUG   = "photometric"
# 03d(서브셋 55%·12에폭)에서 best epoch 이 **5**, 조기 종료가 10에폭이었습니다.
# 풀 데이터는 1.8배라 best 가 뒤로 밀리겠지만 5배까지는 아닙니다.
# patience=5 라 18이면 여유가 있고, 더 필요하면 마지막 에폭이 best 로 찍혀 알려줍니다.
STAGE1_EPOCHS = 18

# 첫 실측 기준선 (conservative bb×0.1 / 1단계 8ep / 2단계 12ep / 배치 16).
# 다시 돌릴 필요 없습니다 — 아래 게이트들이 이 값과 비교해 개선폭을 보여줍니다.
# 근거: docs/results/STEP4A_베이스라인_실측.md
BASELINE = {"stage1_auroc": 0.8031, "stage2_macro_f1": 0.4865, "stage2_a6_recall": 0.328}

# 실측 참고값. 이번 실행이 여기서 크게 벗어나면 데이터·환경을 먼저 의심하세요.
# ⚠️ 1단계 기준값은 **f320 서브셋(STEP 9-A)** 입니다. 풀 데이터라 조금 더 오를 수 있습니다.
MEASURED_384 = {"stage1_auroc": 0.9477,      # f320 · 서브셋 12에폭 (2026-08-23, 03e)
                "stage1_blur_drop": 0.031,   # f320 — full 은 11.1% 였습니다
                "stage2_macro_f1": 0.5395,   # m2.5 (2026-08-22, 03c)
                "stage2_scale_drop": 0.188}  # m2.5

# ── 붙어 있는 크롭 확인 — 학습 전에 여기서 걸러야 합니다 ─────────────
_tags = crop.available_tags()
print(f"붙어 있는 크롭 태그: {_tags}")
if BEST_CROP not in _tags:
    raise SystemExit(
        f"❌ 2단계 크롭 '{BEST_CROP}' 이 없습니다. 붙어 있는 것: {_tags}\n"
        f"   Kaggle 이면 [Add Input] 으로 그 크롭 데이터셋을 붙이세요.")
# ★ STEP 9-A (2026-08-23): 1단계 크롭을 f320 으로 확정했습니다.
#   val AUROC 0.8272(full) → 0.9477(f320), 흐림 하락 11.1% → 3.1%.
#   근거: docs/results/STEP9A_1단계_f320_실측.md
#   choose_stage1_tag() 가 f320 을 full 보다 먼저 고르므로, 붙여만 두면 됩니다.
if not any(crop.fixed_of_tag(t) for t in _tags):
    print("\n🚨 f320 (고정 픽셀) 크롭이 안 붙어 있습니다.")
    print("   1단계는 f320 으로 확정됐습니다 (STEP 9-A: AUROC +0.12, 흐림 하락 −8%p).")
    if "full" in _tags:
        print("   ⚠️ full 로 떨어져서 **멈추지 않고 조용히 나빠집니다.**")
        print("      지금 [Add Input] 으로 dogskin-f320 을 붙이고 다시 돌리세요.\n")
    else:
        print("   full 도 없어서 18번 셀에서 멈춥니다 — dogskin-f320 을 붙이세요.\n")

d = crop.switch_tag(df, BEST_CROP)
report = crop.audit(d, cfg=CFG(img_size=IMG_SIZE))

### 🚦 감사 결과 읽기

| 결과 | 의미 | 대응 |
|---|---|---|
| `[1](a)` 정상/병변 배율 **1.5배 이상** | 크롭 배율이 정상/이상을 흘림 | **1단계를 `full` 크롭으로** (자동) |
| `[1](b)` 병변 6종 간 배율 **2배 이상** | 크롭 배율이 병변 종류를 흘림 | 고정 픽셀 크롭 필요 (아래 판단) |
| `[2]` 확대 비율이 50% 넘음 | 없는 디테일을 만들어 냄 | `[1](b)` 와 같은 원인 |
| `[3]` 흐림 비율 30% 넘음 | 원본 사진 품질 한계 | 실사용에서 흐린 사진 거절 (05) |
| `[3]` 정상/병변 선명도 격차 | 배율 차이의 **부작용** | `[1]` 을 고치면 함께 완화됨 |
| `[4]` 라벨 충돌 > 0 | 같은 파일명에 다른 라벨 | 멈추고 알려주세요 |
| `[5]` 이탈량 50px 미만 | 라벨러 오차 | 무시 가능 (크롭이 알아서 잘라 넣음) |
| `[5]` 이탈량 50px 이상 | 좌표 해석 오류 | 멈추고 알려주세요 |

`[1]`과 `[3]`은 **같은 원인**입니다: 박스가 작으면 → 더 확대되고 → 흐려집니다.
따로 고칠 문제가 아닙니다.

### `full` 크롭의 대가 — 병변이 화면 밖으로 나가는 비율

배율 지름길을 막는 가장 확실한 방법은 박스를 아예 안 쓰는 `full`(중앙 정사각) 크롭입니다.
그런데 **병변이 좌우 끝에 있으면 화면에서 잘려 나갑니다.**
그 사진은 "이상"인데 정상처럼 보이니, **1단계 recall의 천장이 데이터 때문에 낮아집니다.**

목표가 recall 0.95인데 천장이 0.90이면 `full`은 못 씁니다. 미리 재고 정합니다.

In [ ]:
loss_full = crop.full_crop_loss(df, "full", cfg=CFG(img_size=IMG_SIZE))

### 얼마나 심각한가 — 사진을 안 보고 맞혀보기 ★

배율 차이가 있다는 건 알았습니다. 그런데 **그게 실제로 얼마나 정답을 흘리는지**는
따로 재야 합니다. 방법은 간단합니다: **픽셀을 한 장도 안 보고** 박스 크기·모양만으로
분류기를 학습시켜 봅니다.

거기서 나오는 점수가 CNN 이 넘어야 하는 **하한선**입니다.

```
CNN macro-F1 0.45  vs  메타데이터만 0.40   →  피부에서 얻은 건 0.05 뿐  🚨
CNN macro-F1 0.45  vs  메타데이터만 0.18   →  대부분 피부에서 얻음     ✅
```

같은 `fold` 를 쓰므로 4번의 CNN 점수와 직접 비교할 수 있습니다.

In [ ]:
# ★ 판단 기준: 크롭 배율로 **이미지에 실제로 보이는** 특징만 씁니다.
floor = crop.shortcut_baseline(d, cfg=CFG(img_size=IMG_SIZE), features="scale_only")

# 참고: 메타데이터 전체를 넣으면 얼마나 나오는지 (종횡비·병변개수·해상도 포함).
# 그것들은 크롭에 안 보이므로 f320 판단에 쓰면 안 됩니다 — 데이터 성질 참고용.
floor_all = crop.shortcut_baseline(d, cfg=CFG(img_size=IMG_SIZE), features="all",
                                   verbose=False)
print(f"\n(참고) 메타데이터 전체 기준: 1단계 AUROC "
      f"{floor_all.get('stage1_auroc_metadata_only', float('nan')):.4f}, "
      f"2단계 macro-F1 {floor_all.get('stage2_macro_f1_metadata_only', float('nan')):.4f}")
print("  이 값이 위보다 높다면, 크롭에 안 보이는 정보(병변 개수 등)가 라벨과")
print("  상관이 있다는 뜻입니다. CNN 은 그걸 못 쓰므로 판단에는 위 숫자를 쓰세요.")

### 판단: 크롭을 다시 만들어야 하나?

위 하한선을 보고 정합니다.

| 하한선 (2단계 macro-F1) | 판단 | 할 일 |
|---|---|---|
| **< 0.30** | 배율 지름길이 약함 | 그냥 진행. 4번에서 CNN 이 하한선을 넘는지 확인 |
| **≥ 0.30** | 배율이 정답을 크게 흘림 | **고정 픽셀 크롭을 만드세요** ↓ |

고정 픽셀 크롭(`f320`)은 병변 중심에서 **항상 320px**을 잘라냅니다.
피부 1mm 가 항상 같은 픽셀 수라 배율로 맞히는 경로가 막힙니다.
대신 큰 병변은 창을 넘어 잘립니다 — 그게 대가입니다.

**로컬 PC 에서** (원본이 있어야 합니다):

```cmd
py prepare_local.py --chunk VL01 --margins -320
py prepare_local.py --finalize
py prepare_local.py --package
```

기존 크롭은 그대로 두고 `f320` 태그만 추가되므로, 올린 뒤 `crop.available_tags()` 에
`f320` 이 보이면 6번의 크롭 비교에 자동으로 포함됩니다.

> 💡 하한선이 애매하면(0.25~0.30) 일단 진행하세요. 4번에서 CNN 점수와 비교한 뒤
> 격차가 작으면 그때 다시 만들면 됩니다. 지금 확실히 아는 건 하한선뿐입니다.

### 눈으로 볼 것 — 딱 하나

아래 표는 클래스마다 한 줄씩입니다. 병변인지 아닌지 판단하지 마세요.
**줄끼리 서로 달라 보이는지만** 보세요.

- 줄마다 달라 보인다 → 모델이 배울 신호가 있습니다 ✅
- 전부 똑같은 털 사진처럼 보인다 → 모델도 구분 못 할 가능성이 큽니다 ⚠️
  (그렇다고 실패는 아닙니다. 모델은 사람이 못 보는 질감 차이를 봅니다.
   다만 기대치를 낮추고, 4번의 macro-F1 을 냉정하게 보셔야 합니다)
- 한 줄 안에서 제각각이다 → 라벨이 섞였을 수 있습니다
- 개 피부/털이 아니라 사람 손·바닥만 보인다 → 크롭이 어긋난 것

In [ ]:
crop.contact_sheet(d, per_class=6)

### 참고: 박스가 어디에 얹혔는지

좌표계가 뒤집혔거나(x↔y) 스케일이 어긋났으면 여기서 드러납니다.
박스가 **크롭 가운데를 크게 차지하는 게 정상**입니다 (margin 1.5 → 폭의 약 2/3).
박스가 구석에 처박혀 있거나 화면을 벗어나면 좌표 해석이 틀린 겁니다.

In [ ]:
crop.preview_with_box(d, n=4)

## 2. 2단계 뷰 만들기

같은 매니페스트에서 **두 개의 뷰**를 만듭니다. 데이터를 복사하는 게 아니라
`label` 컬럼만 다르게 보는 겁니다.

```
df  ──▶ to_stage1()  label: A7 / ABNORMAL      전체 45,885행
    └─▶ to_stage2()  label: A1~A6              병변 23,070행만
```

`fold` / `is_holdout` / `group` 은 그대로 따라옵니다. → 두 단계가 **같은 분할**을 씁니다.

> ⚠️ **두 단계가 서로 다른 크롭을 쓸 수 있습니다.** 위 감사에서 정상/병변의 박스
> 배율이 다르게 나왔다면, 1단계는 `full` 크롭을 씁니다 — ROI 크롭이 배율로
> 정답을 흘리기 때문입니다. 분할은 여전히 공유하므로 누수는 생기지 않습니다.

In [ ]:
# ★ 1단계 크롭 결정 — 규칙은 src/crop.py 에 있습니다 (노트북 셀은 git pull 로 안 바뀜).
#
# ⚠️ **1단계는 BEST_CROP 을 따라오지 않습니다.** 여기서 한 번 헷갈렸습니다.
#    2단계는 병변 형태를 보려고 ROI 크롭을 쓰지만, 1단계에서 ROI 크롭을 쓰면
#    크롭 창 크기가 정답을 흘립니다 (정상 bbox 0.71% vs 병변 1.25%).
#    그 신호는 배포에 없으므로(보호자 사진에는 bbox 가 없음) 검증 점수만 부풀립니다.
choice = crop.choose_stage1_tag(
    best_crop=BEST_CROP,
    scale_gap=report.get("area_ratio_normal_over_lesion", 1.0),
    full_ceiling=loss_full.get("stage1_recall_ceiling", 1.0),
    target_recall=CFG().target_recall_stage1,          # 보통 0.95
    tags=crop.available_tags(),
)
STAGE1_CROP = choice["tag"]
print(f"1단계 크롭: '{STAGE1_CROP}'\n  근거: {choice['why']}")
for w in choice["warnings"]:
    print(w)

# 2단계(병변 종류)는 ROI 크롭을 씁니다 — 형태를 구분하려면 병변을 크게 봐야 합니다.
s1_all = stages.to_stage1(crop.switch_tag(df, STAGE1_CROP))
s2_all = stages.to_stage2(d)

# 두 뷰 각각 누수 재확인 — 뷰를 만드는 과정에서 분할이 깨지지 않았는지
split.verify(s1_all, fold=0, strict=True)
split.verify(s2_all, fold=0, strict=True)

### 학습 전 1분 — 에폭이 몇 분 걸릴지, 왜 그런지

여기서부터 GPU 를 1시간 넘게 씁니다. 그 전에 **처리량을 재두면** 두 가지를 압니다:

1. 에폭당 몇 분인가 → 90분을 쓸지 말지 지금 결정할 수 있습니다
2. **무엇이 병목인가** → 느릴 때 어디를 고칠지

병목은 둘 중 하나입니다. 처방이 정반대라서 추측하면 안 됩니다:

| 병목 | 증상 | 처방 | 하면 안 되는 것 |
|---|---|---|---|
| **입력 파이프라인** (CPU·디스크) | 배치를 키워도 안 빨라짐 | 크롭을 작게 저장, CPU 코어 늘리기 | 배치·해상도 조절 (무효) |
| **GPU** | 배치를 키우면 느려짐 | 모델·해상도 줄이기 | 워커 늘리기 (무효) |

> ⚠️ 이 셀은 **학습이 안 돌고 있을 때** 돌려야 정확합니다.
> 학습 중에 돌리면 GPU·CPU 를 나눠 쓰게 되어 둘 다 틀립니다.

In [ ]:
from src import bench, experiments
from src.config import with_finetune

# ── ① 병목이 어디인가 (CPU 로더 vs GPU) ─────────────────────────
_bcfg = with_finetune(CFG(model_name="resnet50", img_size=IMG_SIZE,
                          balance_strategy="class_weight"), "moderate")
perf = bench.report(split.get_fold(s2_all, 0)[0], _bcfg, classes=CLASSES)

# ── ② 총 예상 시간 ──────────────────────────────────────────────
# ⚠️ 두 단계는 **백본도 데이터 크기도 다릅니다.**
#    1단계 effnetv2_s / 정상 포함 (2배 데이터) / STAGE1_EPOCHS
#    2단계 resnet50   / 병변만              / 25에폭
#    예전에는 둘 다 resnet50·1단계 12에폭으로 계산해서 크게 빗나갔습니다.
_n1 = len(split.get_fold(s1_all, 0)[0])
_n2 = len(split.get_fold(s2_all, 0)[0])
_e1 = experiments.estimate_runtime([STAGE1_MODEL], IMG_SIZE, _n1,
                                   STAGE1_EPOCHS, n_conditions=1)
_e2 = experiments.estimate_runtime(["resnet50"], IMG_SIZE, _n2, 25, n_conditions=1)
_tot = _e1["total_hours"] + _e2["total_hours"]
print(f"\n★ 1단계 {_e1['total_hours']:.1f}시간 + 2단계 {_e2['total_hours']:.1f}시간 "
      f"= 학습 총 **{_tot:.1f}시간**")
print("   (+ 교란 검사·holdout·크롭 확인 별도. 조기 종료가 걸리면 줄어듭니다)")
if _tot > 6:
    print(f"\n🚨 {_tot:.1f}시간은 깁니다. STAGE1_EPOCHS 를 낮추는 걸 고려하세요 "
          f"(03d 에서 best epoch 은 5 였습니다).")

## 3. 1단계 — 정상 / 이상

거의 5:5 라 학습이 수월합니다. 대신 **평가 기준이 다릅니다**:
macro-F1 이 아니라 **재현율(recall)** 이 먼저입니다.

> 오탐(정상인데 병원 가보라고 함) = 보호자가 헛걸음
> 미탐(병변인데 괜찮다고 함) = **놓친 병**
>
> 둘의 비용이 전혀 다르므로 recall 을 0.95로 **먼저 고정**하고,
> 그 조건에서 precision 이 얼마나 나오는지를 봅니다.

📖 [`docs/basics/08_확률보정과_임계값_결정.md`](../docs/basics/08_확률보정과_임계값_결정.md)

In [ ]:
from src.config import with_finetune, with_aug

# ★ 털 가중 샘플러 — 03g(STEP 15)에서 채택된 alpha 를 여기 적습니다.
#   0 이면 지금까지와 똑같습니다 (balance_strategy="none").
#   ⚠️ 03g 를 안 돌렸으면 **0 으로 두세요.** 안 재본 걸 켜면 06 결과가
#      "백본 때문인지 샘플러 때문인지" 못 가릅니다.
STAGE1_HAIR_ALPHA = 0.0          # ← 03g 결과에 맞춰 고치세요

_bal = "hair_weighted" if STAGE1_HAIR_ALPHA > 0 else "none"
cfg1 = with_aug(with_finetune(
    CFG(model_name=STAGE1_MODEL, img_size=IMG_SIZE,
        epochs=STAGE1_EPOCHS,
        balance_strategy=_bal,           # 5:5 라 기본은 가중치 불필요
        hair_alpha=STAGE1_HAIR_ALPHA,
        monitor="macro_f1",
        # ⚠️ 샘플러가 다르면 **이름도 달라야** 합니다 — 안 그러면 이전 체크포인트를
        #    "이미 끝난 학습" 으로 착각하고 건너뜁니다
        exp_name=f"stage1_{STAGE1_MODEL}_{STAGE1_CROP}_{IMG_SIZE}"
                 + (f"_hair{STAGE1_HAIR_ALPHA:g}" if STAGE1_HAIR_ALPHA > 0 else "")),
    "moderate"), STAGE1_AUG)             # 2단계와 같은 강도로 (조건을 맞춰 비교)
if STAGE1_HAIR_ALPHA > 0:
    print(f"★ 털 가중 샘플러 켬 (alpha={STAGE1_HAIR_ALPHA:g}) — "
          "클래스 총량은 보존됩니다. 아래 [data] 줄에서 확인하세요.")

tr1, va1 = split.get_fold(s1_all, cfg1.use_fold)
print(f"1단계  {STAGE1_MODEL} / {STAGE1_AUG}  ·  train {len(tr1):,} / val {len(va1):,}")
print(f"  헤드 lr {cfg1.lr:.1e} / 백본 lr {cfg1.lr * cfg1.backbone_lr_mult:.1e}"
      f" / {cfg1.epochs}에폭 / 배치 {cfg1.resolved_batch_size()}")

m1 = models.build(STAGE1_MODEL, n_classes=len(CLASSES_STAGE1),
                  pretrained=True, drop_rate=cfg1.drop_rate)
dl_tr1, dl_va1, ds_tr1, _ = data.build_loaders(tr1, va1, cfg1, model=m1,
                                               classes=CLASSES_STAGE1)

In [ ]:
# 이미 돌린 게 있으면 이어서 / 건너뜁니다 (세션이 끊겨도 여기서 회복됩니다)
train.print_status(cfg1.exp_name)

res1 = train.fit(m1, dl_tr1, dl_va1, cfg1, ds_train=ds_tr1)
res1.plot()

# ✅ fit 은 끝나면 **best 에폭 가중치**를 m1 에 되돌려 놓습니다.
#    (EMA 를 쓰므로 저장된 best 는 EMA 가중치입니다 — 아래 평가와 일치)

In [ ]:
# 검증셋 점수 → '이상일 확률' → recall 0.95 지점의 임계값
# ★ 결과를 캐시합니다 — 세션이 끊겨 다시 돌릴 때 이 추론을 건너뜁니다.
#    모델·데이터·순서·TTA 중 하나라도 바뀌면 자동으로 다시 계산합니다.
lg1_va, y1_va = train.cached_logits(m1, dl_va1, key="val", exp=cfg1.exp_name,
                                    n_cls=len(CLASSES_STAGE1), device=DEV,
                                    tta_hflip=cfg1.tta_hflip)
scores1 = stages.stage1_scores(lg1_va)
ybin1 = stages.binary_targets(y1_va)

bin1 = evaluate.binary_report(scores1, ybin1, target_recall=cfg1.target_recall_stage1)
THR1 = bin1["threshold"]

## 1-b. 1단계 확률 보정 ★ (2026-08-26 추가)

**왜 이제야 하나** — 예전에는 1단계 확률이 임계값과 비교만 됐습니다. 순위만
맞으면 되니까 보정이 필요 없었죠. 그런데 앱 화면이 **"이상 가능성 62%" 를
보호자에게 직접 보여주게** 됐습니다. 보정 안 된 숫자를 사람에게 띄우면 안 됩니다.

**임계값도 다시 뽑습니다.** 온도로 나누면 확률 값이 바뀌니 0.1823 이 그대로면
안 됩니다. 다만 온도 스케일링은 **순위를 안 바꾸므로**(단조 변환) 같은 recall 을
같은 사진들로 달성합니다 — 바뀌는 건 임계값 **숫자**뿐이고 판정은 동일합니다.

⚠️ T 는 **검증셋**으로 맞추고 효과는 holdout 에서 확인합니다.

In [ ]:
# ── 1단계 온도 보정 ─────────────────────────────────────────
T1 = calibrate.fit_temperature(lg1_va, y1_va)

# 보정된 점수로 임계값을 다시 뽑습니다.
# 온도는 단조 변환이라 **순위가 안 바뀌므로** recall 은 그대로이고
# 임계값 숫자만 옮겨갑니다. (판정이 달라지면 그게 버그입니다)
# ⚠️ `calibrate.apply()` 는 **이미 확률**을 돌려줍니다. 거기에 stage1_scores 를
#    걸면 softmax 가 두 번 먹어 전부 0.5 쪽으로 뭉개집니다 (실제로 그렇게 짰다가
#    ECE 가 0.03 → 0.20 으로 나빠져서 잡았습니다). logits 를 T 로 나눠서 넘깁니다.
scores1_cal = stages.stage1_scores(lg1_va / T1)
bin1_cal = evaluate.binary_report(scores1_cal, ybin1,
                                  target_recall=cfg1.target_recall_stage1)
THR1_RAW, THR1 = THR1, bin1_cal["threshold"]

# 같은 사진을 같게 판정하는지 확인 — 다르면 순위가 깨진 것입니다
_same = ((scores1 >= THR1_RAW) == (scores1_cal >= THR1)).mean()
print(f"1단계 온도  T = {T1:.4f}")
print(f"  임계값   {THR1_RAW:.4f} → {THR1:.4f}  (보정 눈금)")
print(f"  판정 일치율 {_same:.4%}   ← 100% 여야 정상 (단조 변환)")
if _same < 0.999:
    print("  ⚠️ 판정이 달라졌습니다. 온도 스케일링이 순위를 바꿀 리 없으니 코드를 보세요.")

# 보정 효과 (검증셋 기준 — holdout 확인은 3절에서)
_p_before = stages.stage1_scores(lg1_va)
_ece_b = calibrate.ece(np.stack([1 - _p_before, _p_before], 1), ybin1)
_ece_a = calibrate.ece(np.stack([1 - scores1_cal, scores1_cal], 1), ybin1)
print(f"  ECE  {_ece_b:.4f} → {_ece_a:.4f}")

# ★ 1단계 체크포인트 **옆에** 저장합니다 — infer.Engine.load 가 여기서 찾습니다.
#   (ck1/name1 은 노트북 후반부에서야 생깁니다. 여기선 방금 학습한 cfg1 을 씁니다)
(train.ckpt_dir(cfg1.exp_name) / "temperature.json").write_text(json.dumps(
    {"temperature": T1, "threshold_calibrated": THR1, "threshold_raw": THR1_RAW,
     "ece_before": _ece_b, "ece_after": _ece_a}, indent=2), encoding="utf-8")
train.sync_to_persist(cfg1.exp_name, files=("temperature.json",))


### 🚦 1단계 게이트

- **AUROC ≥ 0.80** 이어야 이진 판정이 의미가 있습니다. 0.5는 동전 던지기입니다.
- recall 0.95 조건에서 precision 이 **0.5 미만**이면 오탐이 절반 넘습니다.
  → 쓸 수는 있지만("의심되니 가보세요" 니까) 사용자 신뢰가 빨리 깎입니다. 개선 대상입니다.

In [ ]:
# ★ 판단 기준은 src/gates.py 에 있습니다 — 노트북 셀과 달리 git pull 로 갱신됩니다.
#   STOP(깨진 수준)에서만 멈추고, WANT(기대치) 미달은 경고 후 진행합니다.
from src import gates

gates.stage1(
    auroc=bin1["auroc"],
    threshold=THR1,
    precision=bin1["precision_at_target"],
    floor=floor.get("stage1_auroc_metadata_only"),
    baseline=BASELINE["stage1_auroc"],
    crop_tag=STAGE1_CROP,
    epochs=cfg1.epochs,
)

## 4. 2단계 — 병변 6종

여기가 어려운 쪽입니다. 불균형이 **6.4배**(A2 5,275 ↔ A5 820)이고,
병변 형태끼리 실제로 닮았습니다.

### 파인튜닝 강도를 올려서 갑니다

용어부터. 자주 헷갈리는 지점입니다:

```
전이학습 (transfer learning)   ← 우산 개념
  ├─ linear probe   백본 얼리고 헤드만 학습
  └─ fine-tuning    백본까지 같이 학습     ← 이 리포는 처음부터 이쪽
```

`freeze_backbone()` 은 정의만 되어 있고 **어디서도 호출하지 않습니다.**
그래서 질문은 "파인튜닝을 할까?" 가 아니라 **"얼마나 세게 할까?"** 입니다.

첫 실측(`conservative`, 12에폭)에서 백본 lr 이 `3e-4 × 0.1 = 3e-5` 였는데,
그 결과가 이랬습니다:

| 증상 | 값 | 해석 |
|---|---|---|
| train vs val loss | 1.352 vs 1.474 | 격차 작음 = **과적합 아님** |
| 12에폭 전부 best 갱신 | val loss 한 번도 안 오름 | **수렴 전** |
| macro AUROC vs F1 | 0.8155 vs 0.4865 | 순위는 좋은데 결정이 나쁨 |

**학습 데이터조차 잘 못 맞춥니다.** ImageNet 은 *물체*를 구분하도록 배웠고
우리 과제는 *피부 질감·색의 미세한 차이*라 도메인 격차가 큽니다.
백본이 적응할 시간과 학습률이 필요합니다.

→ 그래서 처음부터 **`moderate`(백본 lr ×0.3) + 25에폭**으로 갑니다.
   비교 기준은 아래 `BASELINE` 에 적어둔 첫 실측값입니다 — 다시 안 돌려도 됩니다.

📖 [`docs/results/STEP4A_베이스라인_실측.md`](../docs/results/STEP4A_베이스라인_실측.md)

In [ ]:
from src.config import with_finetune

cfg2 = with_finetune(
    CFG(model_name="resnet50", img_size=IMG_SIZE,
        epochs=25,                       # 12에폭 전부 best 갱신이었음 → 수렴까지
        balance_strategy="class_weight", # 6.4배 불균형
        monitor="macro_f1",
        exp_name=f"stage2_resnet50_{BEST_CROP}_{IMG_SIZE}"),     # ★ 해상도 포함
    "moderate")                          # 백본 lr ×0.1 → ×0.3

tr2, va2 = split.get_fold(s2_all, cfg2.use_fold)
print(f"2단계  train {len(tr2):,} / val {len(va2):,}")
print(f"  헤드 lr {cfg2.lr:.1e} / 백본 lr {cfg2.lr * cfg2.backbone_lr_mult:.1e}"
      f" / {cfg2.epochs}에폭 / 배치 {cfg2.resolved_batch_size()}")

m2 = models.build("resnet50", n_classes=len(CLASSES),
                  pretrained=True, drop_rate=cfg2.drop_rate)
dl_tr2, dl_va2, ds_tr2, _ = data.build_loaders(tr2, va2, cfg2, model=m2, classes=CLASSES)

In [ ]:
# 증강이 실제로 뭘 하는지 눈으로 보기
import matplotlib.pyplot as plt
from src.data import IMAGENET_MEAN, IMAGENET_STD

x, y = next(iter(dl_tr2))
mean = torch.tensor(IMAGENET_MEAN).view(3,1,1); std = torch.tensor(IMAGENET_STD).view(3,1,1)
fig, axes = plt.subplots(2, 4, figsize=(12, 6.2))
for ax, i in zip(axes.flat, range(min(8, len(x)))):
    ax.imshow((x[i]*std+mean).clamp(0,1).permute(1,2,0)); ax.axis("off")
    ax.set_title(f"{CLASSES[y[i]]} {CLASS_KO[CLASSES[y[i]]][:8]}", fontsize=8)
plt.suptitle("증강 후 실제로 모델이 보는 이미지"); plt.tight_layout(); plt.show()
print("💡 병변이 잘려 나가거나 색이 심하게 변했다면 증강이 너무 센 겁니다.")

In [ ]:
train.print_status(cfg2.exp_name)

res2 = train.fit(m2, dl_tr2, dl_va2, cfg2, ds_train=ds_tr2)
res2.plot()

### 학습 곡선 읽는 법

| 증상 | 의미 | 대응 |
|---|---|---|
| train↓ val↓ 둘 다 계속 하락 | 정상, 더 학습 가능 | epochs 늘리기 |
| train↓ **val↑** | 과적합 시작 | 조기종료 지점, 증강↑ / drop_rate↑ |
| 둘 다 안 내려감 | 학습이 안 됨 | lr 조정, 데이터/라벨 확인 |
| val 이 심하게 출렁임 | 배치가 작거나 lr 이 큼 | batch↑ 또는 lr↓ |

In [ ]:
lg2_va, y2_va = train.cached_logits(m2, dl_va2, key="val", exp=cfg2.exp_name,
                                    n_cls=len(CLASSES), device=DEV,
                                    tta_hflip=cfg2.tta_hflip)
rep2 = evaluate.full_report(lg2_va, y2_va, CLASSES)
rep2.plot_confusion()
rep2.plot_per_class()

### 🚦 2단계 게이트

**macro-F1 이 0.25 미만이면 여기서 멈추고 데이터를 다시 보세요.**
랜덤이 1/6 ≈ 0.167 인데 그것보다 조금 나은 수준이면 파이프라인 어딘가가 깨진 겁니다.

흔한 원인: 라벨 매칭 오류, 크롭 좌표 오류, 클래스 매핑 뒤바뀜.

In [ ]:
from src import gates

cnn_f1 = rep2.metrics["macro_f1"]
a6_recall = rep2.metrics["per_class"]["recall"][CLASSES.index("A6")]

gates.stage2(
    macro_f1=cnn_f1,
    floor=floor.get("stage2_macro_f1_metadata_only"),
    baseline=BASELINE["stage2_macro_f1"],
)

# ── ② 첫 실측 대비: 파인튜닝 강도를 올린 효과 ────────────────────
print(f"\n[첫 실측 대비]  bb×0.1/12ep  →  bb×{cfg2.backbone_lr_mult}/{cfg2.epochs}ep")
print(f"  macro-F1   {BASELINE['stage2_macro_f1']:.4f} → {cnn_f1:.4f}"
      f"   ({cnn_f1 - BASELINE['stage2_macro_f1']:+.4f})")
print(f"  A6 recall  {BASELINE['stage2_a6_recall']:.3f} → {a6_recall:.3f}"
      f"   ({a6_recall - BASELINE['stage2_a6_recall']:+.3f})")

# ── ③ 클래스별 하한선 — 지름길은 평균에 안 보입니다 ──────────────
# ⚠️ 차이의 **부호**가 뜻이 정반대입니다:
#     차이 > 0 인데 작다 → 크기 정보에 얹혀 있을 수 있음 (지름길 의심)
#     차이 < 0           → 크기보다도 못함 = 크기를 **안** 쓰는 중.
#                          지름길이 아니라 그 병변 자체가 어려운 것입니다.
base_rec = floor.get("stage2_recall_metadata_only") or {}
if base_rec:
    cnn_rec = rep2.metrics["per_class"]["recall"]
    print(f"\n  {'클래스':<8}{'하한선':>9}{'CNN':>9}{'차이':>9}   판정")
    leaky, weak = [], []
    for i, c in enumerate(CLASSES):
        b, v = base_rec.get(c, 0.0), cnn_rec[i]
        d = v - b
        if b > 0.3 and 0 <= d < 0.15:
            note = "⚠️ 크기에 얹혀 있을 수 있음"; leaky.append(c)
        elif d < 0:
            note = "→ 크기를 안 씀. 이 병변이 어려운 것"; weak.append(c)
        else:
            note = ""
        print(f"  {c:<8}{b:>9.3f}{v:>9.3f}{d:>+9.3f}   {note}")
    if leaky:
        print(f"\n  ⚠️ 지름길 의심: {', '.join(leaky)} → 6번 배율 교란 검사로 확인")
    if weak:
        print(f"\n  📉 크기보다도 못한 클래스: {', '.join(weak)}")
        print("     지름길 문제가 **아닙니다** — 재크롭으로 해결되지 않습니다.")
        print("     원인은 학습 부족 / 표본 부족 / 병변 난이도입니다.")

# ── ④ 수렴했는가 — 에폭을 더 줘야 하나 ──────────────────────────
h = res2.history
if res2.best_epoch >= len(h) - 2:
    print(f"\n  📈 마지막 에폭({res2.best_epoch})이 최고 — 아직 수렴하지 않았습니다.")
    print(f"     위 cfg2 의 epochs 를 {cfg2.epochs} → {int(cfg2.epochs * 1.6)} 로 올려 다시 돌려보세요.")
else:
    print(f"\n  ✅ epoch {res2.best_epoch} 에서 최고 후 개선 없음 — 수렴했습니다.")
    print("     에폭을 더 늘려도 이 설정으로는 안 오릅니다.")

## 5. 두 단계를 이어붙이기 ★ 여기가 진짜 성능

각 단계를 따로 잘 하는 것과, **이어붙여서** 잘 하는 것은 다릅니다.
**1단계가 놓친 병변은 2단계가 볼 기회조차 없습니다.**
사용자가 실제로 겪는 건 이 이어붙인 결과입니다.

```
사진 → 1단계 ─ '이상 확률' < 임계값 → "정상으로 보입니다"   (2단계는 안 봄)
              └ 임계값 이상 ────────→ 2단계 → "A2 소견이 의심됩니다"
```

⚠️ 평가할 때 중요한 점: **2단계 모델도 정상 사진에 돌려야 합니다.**
실제 서비스에서는 정상 사진도 1단계를 통과하면 2단계로 넘어오니까요.
그래서 두 모델을 **같은 검증셋(정상 포함), 같은 순서**로 돌립니다.

In [ ]:
# 전체 검증셋 = 정상 + 병변. 두 모델을 같은 행·같은 순서로 돌립니다.
va_all = split.get_fold(s1_all, cfg1.use_fold)[1]        # 1단계 뷰 (label_orig 보존)

# ⚠️ 두 단계가 다른 크롭을 쓸 수 있습니다 (1단계 full / 2단계 m1.5).
#    각 모델에는 **그 모델이 학습한 크롭**을 먹여야 합니다.
#    switch_tag 는 image_path 해시로 경로를 다시 계산하므로 행 순서가 보존됩니다.
va_s1 = va_all
va_s2 = crop.switch_tag(va_all, BEST_CROP, verbose=False) if STAGE1_CROP != BEST_CROP \
        else va_all
print(f"1단계 입력 크롭: {STAGE1_CROP}  |  2단계 입력 크롭: {BEST_CROP}")

dl_e1, ds_e1 = data.eval_loader(va_s1, cfg1, model=m1, classes=CLASSES_STAGE1)
dl_e2, ds_e2 = data.eval_loader(va_s2, cfg2, model=m2, classes=CLASSES)

# 순서가 어긋나면 점수가 조용히 엉망이 됩니다 — 반드시 확인
assert len(ds_e1.df) == len(ds_e2.df), \
    f"행 수가 다릅니다: {len(ds_e1.df)} vs {len(ds_e2.df)} — 한쪽 크롭이 빠졌습니다"
assert (ds_e1.df["image_name"].to_numpy() == ds_e2.df["image_name"].to_numpy()).all(), \
    "두 로더의 행 순서가 다릅니다"

# 전체 검증셋(정상 포함) 두 번 — 여기가 노트북에서 가장 무거운 추론입니다. 캐시합니다.
lg1_e, _ = train.cached_logits(m1, dl_e1, key=f"pipe_{STAGE1_CROP}", exp=cfg1.exp_name,
                               n_cls=len(CLASSES_STAGE1), device=DEV, tta_hflip=True)
lg2_e, _ = train.cached_logits(m2, dl_e2, key=f"pipe_{BEST_CROP}", exp=cfg2.exp_name,
                               n_cls=len(CLASSES), device=DEV, tta_hflip=True)

s1_sc = stages.stage1_scores(lg1_e)
y_final = ds_e1.df["label_orig"].to_numpy()               # A1~A7 원래 라벨
print(f"평가 대상 {len(y_final):,}장 (정상 {(y_final == NORMAL_LABEL).sum():,} / "
      f"병변 {(y_final != NORMAL_LABEL).sum():,})")

In [ ]:
pipe = stages.pipeline_report(s1_sc, lg2_e, y_final, threshold=THR1)
stages.plot_pipeline_confusion(pipe)

## 6. 실사용 견고성 검사 ★ 여기서 진짜가 드러납니다

지금까지의 점수는 모두 **우리가 만든 크롭** 위에서 잰 것입니다.
그 크롭은 병변을 정중앙에 두고, 병변 크기에 맞춰 배율을 정했습니다.
보호자 사진은 둘 다 아닙니다.

그래서 검증셋을 일부러 그렇게 망가뜨려 보고 점수 하락폭을 잽니다.
**하락폭이 곧 실사용 위험도**입니다. 학습은 안 하니 몇 분이면 됩니다.

| 하락폭 | 판정 |
|---|---|
| 15% 미만 | ✅ 견고 |
| 15~30% | ⚠️ 상당히 의존 — 개선 여지 큼 |
| 30% 이상 | 🚨 실사용에서 무너짐 |

> 💡 **`f320`의 효과는 하한선이 아니라 이 숫자로 판정하세요.**
> 하한선(`shortcut_baseline`)은 `bbox` 컬럼을 쓰기 때문에 크롭 방식을 바꿔도
> 거의 안 변합니다. "데이터에 상관이 있나"와 "모델이 그걸 썼나"는 다른 질문입니다.

In [ ]:
from src import robust

rb = robust.report(m2, va2, cfg2, CLASSES, n=2000)

### 임계값을 바꾸면 무엇이 바뀌나

임계값 하나가 이 시스템의 **성격**을 정합니다.
낮추면 놓치는 병변은 줄고 헛알림이 늘어납니다.

이 프로젝트의 목적("의심된다까지 알려주기")에서는 **놓치지 않는 쪽**이 맞습니다.
다만 헛알림이 너무 많으면 보호자가 알림을 무시하게 되므로, 표를 보고 균형점을 잡으세요.

In [ ]:
import pandas as pd
import numpy as np

rows = []
for t in np.quantile(s1_sc, [0.05, 0.15, 0.25, 0.35, 0.45, 0.55, 0.65]):
    r = stages.pipeline_report(s1_sc, lg2_e, y_final, threshold=float(t), show=False)
    rows.append({"임계값": round(float(t), 4),
                 "놓친병변": r["lesion_missed"],
                 "스크리닝recall": round(r["lesion_screening_recall"], 4),
                 "헛알림비율": round(r["false_alarm_rate"], 4),
                 "종류정확도": round(r["kind_accuracy_given_routed"], 4),
                 "최종macroF1": round(r["final_macro_f1"], 4)})
tbl = pd.DataFrame(rows)
print(tbl.to_string(index=False))
print(f"\n선택한 임계값: {THR1:.4f} (recall {cfg1.target_recall_stage1:.0%} 목표 기준)")
print("💡 '스크리닝recall' 이 0.95를 넘는 가장 큰 임계값을 고르면 헛알림이 최소가 됩니다.")

## 7. 결과 저장 — 여기까지가 필수입니다

노트북 04·05 가 읽어갈 선택 결과를 남깁니다.
아래 8번(해상도 실험)이 끝나면 이 셀을 다시 돌려 최종본으로 갱신하세요.

In [ ]:
import json

W = env.work_root()
W.mkdir(parents=True, exist_ok=True)

(W/"best_crop.txt").write_text(BEST_CROP)
(W/"stage1_threshold.json").write_text(json.dumps({
    "threshold": THR1,
    "target_recall": cfg1.target_recall_stage1,
    "auroc": bin1["auroc"],
    "precision_at_target": bin1["precision_at_target"],
    "stage1_crop": STAGE1_CROP,        # ← 1단계는 다른 크롭일 수 있습니다
    "stage2_crop": BEST_CROP,
    # ★ 실험 이름을 남깁니다. with_finetune 이 프리셋 이름을 붙이므로
    #   (stage1_resnet50_full → stage1_resnet50_full_moderate) 노트북 05 가
    #   이름을 추측하면 못 찾습니다.
    "stage1_exp": cfg1.exp_name,
    "stage2_exp": cfg2.exp_name,
    "img_size": IMG_SIZE,
    "finetune": {"backbone_lr_mult": cfg2.backbone_lr_mult, "epochs": cfg2.epochs},
    "audit": {k: v for k, v in report.items() if not isinstance(v, (dict, list))},
}, indent=2, ensure_ascii=False))

summary = {
    "stage1": {"crop": STAGE1_CROP, "auroc": bin1["auroc"], "threshold": THR1,
               "model": STAGE1_MODEL, "aug": STAGE1_AUG,
               "epochs": cfg1.epochs, "backbone_lr_mult": cfg1.backbone_lr_mult},
    "stage2": {"crop": BEST_CROP, "macro_f1": rep2.metrics["macro_f1"],
               "ci": list(rep2.ci[1:]),
               "per_class_recall": dict(zip(CLASSES, rep2.metrics["per_class"]["recall"])),
               "epochs": cfg2.epochs, "backbone_lr_mult": cfg2.backbone_lr_mult},
    "floor": {"stage1_auroc": floor.get("stage1_auroc_metadata_only"),
              "stage2_macro_f1": floor.get("stage2_macro_f1_metadata_only")},
    "pipeline": {k: v for k, v in pipe.items() if not isinstance(v, (dict, list))},
    "robustness": {"scale_rel_drop": rb["scale"].get("_summary", {}).get("rel_drop"),
                   "shift_rel_drop": rb["shift"].get("_summary", {}).get("rel_drop")},
    "baseline_first_run": BASELINE,
    "exp_names": {"stage1": cfg1.exp_name, "stage2": cfg2.exp_name},
    "img_size": IMG_SIZE,
}
(W/"reports").mkdir(parents=True, exist_ok=True)
(W/"reports"/"step4a_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False))

# ★ 다음 노트북(04/05)에 넘길 것만 한 폴더로 모읍니다.
#   work/ 안쪽에 두면 Kaggle 이 데이터셋으로 만들 때 빠집니다 (실제로 빠졌습니다).
train.export_release(
    exps=[cfg1.exp_name, cfg2.exp_name],
    meta={"1단계 크롭": STAGE1_CROP, "2단계 크롭": BEST_CROP, "입력": f"{IMG_SIZE}px",
          "1단계 AUROC": f"{bin1['auroc']:.4f}", "임계값": f"{THR1:.4f}",
          "2단계 macro-F1": f"{rep2.metrics['macro_f1']:.4f}",
          "배율 하락": f"{rb['scale'].get('_summary', {}).get('rel_drop', float('nan')):.1%}"},
    files={"stage1_threshold.json": json.loads((W/"stage1_threshold.json").read_text()),
           "reports/step4a_summary.json": summary},
)

print("저장 완료")
print(f"  1단계 크롭 {STAGE1_CROP} / 2단계 크롭 {BEST_CROP}")
print(f"  work_root: {W}")
print("\n" + "=" * 60)
print(" STEP 4A 결과 (이 블록을 복사해서 공유하세요)")
print("=" * 60)
print(f"  1단계 AUROC        {bin1['auroc']:.4f}   (첫 실측 {BASELINE['stage1_auroc']:.4f})")
print(f"  2단계 macro-F1     {rep2.metrics['macro_f1']:.4f}   (첫 실측 {BASELINE['stage2_macro_f1']:.4f})")
print(f"  A6 recall          {rep2.metrics['per_class']['recall'][CLASSES.index('A6')]:.3f}"
      f"   (첫 실측 {BASELINE['stage2_a6_recall']:.3f})")
print(f"  스크리닝 recall    {pipe['lesion_screening_recall']:.4f}")
print(f"  헛알림 비율        {pipe['false_alarm_rate']:.4f}")
print(f"  배율 교란 하락     {rb['scale'].get('_summary', {}).get('rel_drop', float('nan')):.1%}")
print(f"  위치 교란 하락     {rb['shift'].get('_summary', {}).get('rel_drop', float('nan')):.1%}")
print("=" * 60)

---
## ✅ 정리

| 확인한 것 | 어디서 | 통과 기준 |
|---|---|---|
| 크롭이 배율로 정답을 흘리지 않는가 | 1번 감사 | 정상/병변 1.5배 미만 |
| `full` 로 바꿔도 병변을 안 잃는가 | 1번 `full_crop_loss` | 천장 ≥ 0.95 |
| 사진 없이 얼마나 맞히는가 | 1번 `shortcut_baseline` | 2단계 < 0.30 |
| 1단계가 정상/이상을 구분한다 | 3번 | AUROC > 0.80 |
| 2단계가 하한선을 넘는다 | 4번 | 차이 > 0.15 |
| **이어붙인 실제 성능** | 5번 | 스크리닝 recall ≥ 0.95 |
| **실사용에서 버티는가** | 6번 | 배율·위치 하락 < 15% |

## 다음 단계

**`03b_증강_배율강건성.ipynb`** — 2단계에 남은 배율 하락 20.4% 를 확대 증강으로 잡습니다.

384 에서 최악 조건이 **축소(0.5x) → 확대(2x)** 로 바뀌었습니다. 축소는 픽셀이 사라지는
물리 문제라 해상도로만 풀리지만, 확대는 훈련 분포 문제라 **증강으로 풀 수 있습니다.**

⚠️ **`04`(백본 비교)는 배율 하락이 잡히기 전까지 보류합니다.**
무너지는 기준 위에서 6개 모델을 비교하면 "배율을 가장 잘 읽는 모델" 을 뽑게 됩니다.
그건 실사용에서 가장 먼저 무너지는 모델입니다.

📖 [`docs/basics/09_ViT와_최신_백본_지도_2026.md`](../docs/basics/09_ViT와_최신_백본_지도_2026.md) ·
[`docs/cautions/08_2단계_파이프라인_설계_주의점.md`](../docs/cautions/08_2단계_파이프라인_설계_주의점.md) ·
[`docs/results/STEP4A_베이스라인_실측.md`](../docs/results/STEP4A_베이스라인_실측.md)


In [ ]:
import json
import numpy as np
import pandas as pd
import torch
from src import (labels, split, crop, data, models, train, evaluate,
                 calibrate, explain, infer, stages)
from src.config import (MODEL_BY_KEY, CLASSES_STAGE1, NORMAL_LABEL,
                        ADOPTED_STAGE2_CROP)

# ★ GPU 없이 진행하면 20~30배 느립니다. 없으면 여기서 멈춥니다.
#   (Colab 무료 한도를 넘기면 말없이 CPU 런타임을 줍니다 — 이걸 막습니다)
env.require_gpu()
DEV = "cuda" if torch.cuda.is_available() else "cpu"
W = env.work_root()

# 앞 노트북들이 남긴 선택 결과를 읽습니다.
#   03 → stage1_threshold.json  (임계값·크롭·실험 이름)
#   04 → best_model.json        (우승 백본)
# 04 를 건너뛰고 03 → 05 로 바로 와도 동작해야 합니다.
def _load(name):
    p = W/name
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}

thr = _load("stage1_threshold.json")        # 노트북 03
crp = _load("reports/step4c_crop.json")     # 노트북 03c (크롭 확정)
sel = _load("best_model.json")              # 노트북 04

# ★ JSON 이 안 넘어와도 체크포인트 **이름**에는 설정이 적혀 있습니다.
#   (stage1_resnet50_full_moderate → 1단계 / resnet50 / full 크롭)
#   가중치는 무거워서 잘 넘어오는데 JSON 은 가벼워서 잘 빠집니다 — 실제로 그랬습니다.
from src import train as _t

inf = _t.infer_run_settings()
if inf and not thr:
    print(f"ℹ️ stage1_threshold.json 이 없어 체크포인트 이름에서 설정을 되살립니다: {inf}")
if not thr and not sel and not inf:
    # ⚠️ 03 을 안 돌린 게 아니라 **인계가 안 된** 경우가 대부분입니다.
    #    Kaggle 은 노트북마다 세션이 따로라 03 의 출력을 입력으로 붙여야 합니다.
    from src import train as _t

    raise FileNotFoundError(
        f"{W} 에 stage1_threshold.json 도 체크포인트도 없습니다.\n"
        "03 을 돌렸다면 그 **출력을 이 노트북 입력으로 붙이지 않은** 것입니다.\n"
        + _t.explain_handoff()
    )

BEST_MODEL  = sel.get("stage2") or inf.get("stage2_model") or (
    "resnet50" if thr else "convnextv2_base")

# 크롭은 03c 가 정합니다 (STEP 4C: m2.5 채택 — 배율 하락 23.2% → 18.8%).
# ⚠️ 03 이 남긴 stage2_crop 은 **일부러 안 봅니다.** 03 은 크롭 비교 이전에 돌아서
#    아직 m1.5 라고 적어두고 있습니다. 낡은 값이 새 결론을 덮어쓰면 안 됩니다.
BEST_CROP   = (sel.get("stage2_crop") or crp.get("best_crop")
               or inf.get("stage2_crop") or "m2.5")
STAGE1_CROP = (sel.get("stage1_crop") or thr.get("stage1_crop")
               or inf.get("stage1_crop") or BEST_CROP)
IMG_SIZE    = int(sel.get("img_size", thr.get("img_size", 384)))

# 1단계가 'full'/'f320' 인 것은 **정상입니다** — ROI 크롭을 쓰면 크롭 창 크기가
# 정답을 흘리는데, 그 신호는 배포에 없습니다 (crop.choose_stage1_tag 참고).
# 촬영 가이드는 2단계 모델로 뽑습니다. 1단계는 배율 정보가 없어 기준이 안 됩니다.
if crop.margin_of_tag(STAGE1_CROP):
    print(f"⚠️ 1단계가 ROI 크롭({STAGE1_CROP}) 입니다 — 지름길을 못 막은 상태입니다.\n"
          f"   1단계 점수를 낙관적으로 취급하세요.")
else:
    print(f"1단계 {STAGE1_CROP} / 2단계 {BEST_CROP} — 의도된 조합입니다 "
          f"(촬영 가이드는 2단계 기준).")

# 배율 → 화면 점유율 변환에 쓰입니다. m1.5 면 1x 에서 병변이 화면의 67%,
# m2.5 면 40% — 즉 크롭이 바뀌면 **보호자에게 할 말이 바뀝니다**.
CROP_MARGIN = crop.margin_of_tag(BEST_CROP) or 1.5

# ★ 붙인 체크포인트가 **채택된 크롭의 것**인지 확인합니다.
#   Kaggle 의 Version 목록에서 예전 실행을 고르기 쉬운데, 그대로 진행하면
#   촬영 가이드·보정·임계값이 전부 버린 설정 기준으로 나오고 에러는 안 납니다.
ALLOW_CROP_MISMATCH = False       # 일부러 다른 크롭을 평가할 때만 True
if BEST_CROP != ADOPTED_STAGE2_CROP and not ALLOW_CROP_MISMATCH:
    raise SystemExit(
        f"❌ 2단계 크롭이 '{BEST_CROP}' 입니다 — 채택된 크롭은 "
        f"'{ADOPTED_STAGE2_CROP}' 입니다 (STEP 4C·4D).\n"
        f"   불러온 체크포인트: {inf.get('stage2_exp') or EXP2 or '(이름 불명)'}\n\n"
        "   → **예전 실행의 출력**을 붙였을 가능성이 큽니다.\n"
        "     Kaggle 노트북의 Version 목록에서 m2.5 로 돌린 버전을 찾아\n"
        "     (2시간 18분 / 2단계 macro-F1 0.5313 / 배율 하락 16.0%)\n"
        "     그 버전의 Output 으로 데이터셋을 **다시** 만들어 붙이세요.\n"
        "   → 일부러 다른 크롭을 보는 거라면 이 셀의 ALLOW_CROP_MISMATCH = True")

# ⚠️ 임계값을 기본값 0.5 로 두면 파이프라인 평가가 조용히 **틀립니다**
#    (03 에서 recall 0.95 로 정한 값은 보통 0.2~0.3 입니다).
#    그래서 못 찾으면 0.5 로 때우지 않고, 아래 6번 셀에서 **다시 계산**합니다.
#    임계값은 "검증셋에서 recall 0.95 가 되는 지점" 이라는 정의뿐이라
#    같은 모델·같은 fold 면 03 과 같은 값이 나옵니다 (holdout 오염 아님).
if "stage1_threshold" in sel:
    THR1 = float(sel["stage1_threshold"])
elif "threshold" in thr:
    THR1 = float(thr["threshold"])
else:
    THR1 = None
    print("⚠️ 1단계 임계값을 못 찾았습니다 → 검증셋에서 다시 계산합니다 (약 1분).")

# 실험 이름은 추측하지 않고 03/04 가 남긴 걸 씁니다
# (with_finetune 이 'stage1_resnet50_full' → 'stage1_resnet50_full_moderate' 로 바꿉니다)
EXP1 = sel.get("stage1_exp") or thr.get("stage1_exp") or inf.get("stage1_exp")
EXP2 = sel.get("stage2_exp") or thr.get("stage2_exp") or inf.get("stage2_exp")
print(f"모델 {BEST_MODEL} | 2단계 크롭 {BEST_CROP}(margin {CROP_MARGIN:g}) | 1단계 크롭 {STAGE1_CROP} | "
      f"입력 {IMG_SIZE}px | 임계값 " + ("(재계산 예정)" if THR1 is None else f"{THR1:.4f}"))
print(f"실험 이름: 1단계 {EXP1 or '(미기록 — 이름으로 탐색)'} / 2단계 {EXP2 or '(미기록)'}")

_raw = labels.load(W/"manifests"/"manifest_final.parquet")
df = crop.switch_tag(_raw, BEST_CROP)
s1_all = stages.to_stage1(crop.switch_tag(_raw, STAGE1_CROP))   # 1단계는 다른 크롭일 수 있음
s2_all = stages.to_stage2(df)

_, va1 = split.get_fold(s1_all, 0)       # 정상 포함
tr2, va2 = split.get_fold(s2_all, 0)     # 병변만
ho1 = split.get_holdout(s1_all)          # 정상 포함 holdout
ho2 = split.get_holdout(s2_all)
print(f"val(전체) {len(va1):,} / val(병변) {len(va2):,} / holdout(전체) {len(ho1):,}")

In [ ]:
# 체크포인트 위치는 어느 노트북에서 왔느냐에 따라 다릅니다.
#   노트북 04 를 돌렸다면  checkpoints/s1_{모델}, s2_{모델}
#   노트북 03 만 돌렸다면  checkpoints/stage1_resnet50_{크롭}, stage2_resnet50_{크롭}
# 03 → 05 로 바로 와도 동작하도록 둘 다 찾습니다.
#
# ★ 학습을 다른 세션에서 했다면 로컬(/content)에는 체크포인트가 없습니다.
#   Drive 백업에서 먼저 되살립니다.
def _find_ckpt(*candidates):
    for name in candidates:
        train.restore_from_persist(name, verbose=False)     # Drive → 로컬
        ck = W/"checkpoints"/name/"best.pt"
        if ck.exists():
            return ck, name
    raise FileNotFoundError(
        "체크포인트를 찾지 못했습니다. 노트북 03(또는 04)을 먼저 돌리세요.\n"
        f"  찾아본 곳: {[str(W/'checkpoints'/n/'best.pt') for n in candidates]}\n"
        f"  로컬에 있는 것: {sorted(p.name for p in (W/'checkpoints').glob('*')) if (W/'checkpoints').exists() else '(폴더 없음)'}\n"
        f"  Drive 백업: {sorted(p.name for p in (env.persist_root()/'checkpoints').glob('*')) if env.persist_root() and (env.persist_root()/'checkpoints').exists() else '(없음)'}"
    )

# EXP1/EXP2 가 있으면 그게 정답입니다. 없으면 예전 방식으로 이름을 맞춰 봅니다.
ck1, name1 = _find_ckpt(*[c for c in (EXP1, f"s1_{BEST_MODEL}",
                                      f"stage1_resnet50_{STAGE1_CROP}_moderate",
                                      f"stage1_resnet50_{STAGE1_CROP}") if c])
ck2, name2 = _find_ckpt(*[c for c in (EXP2, f"s2_{BEST_MODEL}",
                                      f"stage2_resnet50_{BEST_CROP}_moderate",
                                      f"stage2_resnet50_{BEST_CROP}") if c])
print(f"1단계 체크포인트: {name1}\n2단계 체크포인트: {name2}")

# ★ 백본은 **체크포인트 폴더 이름에서** 읽습니다.
#   예전에는 "03 의 체크포인트는 항상 resnet50" 으로 하드코딩했는데,
#   STEP 6 에서 1단계를 effnetv2_s 로 바꾸면서 깨졌습니다 — effnetv2_s 가중치를
#   resnet50 껍데기에 부으려다 shape mismatch(bn1 24 vs 64)로 죽었습니다.
#   두 단계가 서로 다른 백본을 쓸 수 있으므로 각각 따로 읽습니다.
_k1 = train.model_key_from_exp(name1) or BEST_MODEL
_k2 = train.model_key_from_exp(name2) or BEST_MODEL
if _k1 not in MODEL_BY_KEY or _k2 not in MODEL_BY_KEY:
    raise SystemExit(
        f"❌ 체크포인트 이름에서 모르는 백본이 나왔습니다: 1단계 '{_k1}' / 2단계 '{_k2}'\n"
        f"   MODEL_ZOO 에 있는 키: {sorted(MODEL_BY_KEY)}\n"
        f"   체크포인트 이름: {name1} / {name2}")
spec1, spec2 = MODEL_BY_KEY[_k1], MODEL_BY_KEY[_k2]
print(f"백본 — 1단계 {_k1} / 2단계 {_k2}")
spec = spec2                       # 이후 cfg 는 2단계 기준
cfg = CFG(model_name=spec.timm_name, img_size=IMG_SIZE, exp_name=name2)

m1 = models.load_checkpoint(str(ck1), spec1, len(CLASSES_STAGE1))
m2 = models.load_checkpoint(str(ck2), spec2, len(CLASSES))
print(f"두 단계 모델 로드 완료  (입력 {cfg.img_size}px)")

# ★ 임계값 복원 — 03 의 JSON 이 안 넘어왔을 때만 돕니다.
#   03 과 **같은 정의·같은 fold·같은 모델**이라 같은 값이 나옵니다.
#   (holdout 은 건드리지 않습니다 — 검증셋만 씁니다)
if THR1 is None:
    _cfg1 = CFG(model_name=spec1.timm_name, img_size=IMG_SIZE, exp_name=name1)
    _dl1, _ds1 = data.eval_loader(va1, _cfg1, model=m1, classes=CLASSES_STAGE1)
    # ⚠️ TTA 를 03 과 맞춰야 같은 임계값이 나옵니다 (03 은 cfg1.tta_hflip=True 로 쟀습니다).
    #    안 맞추면 AUROC 가 0.8155 → 0.8143 처럼 미묘하게 달라집니다.
    _, _lg1, _y1 = train.evaluate_loader(m1, _dl1, None, DEV, len(CLASSES_STAGE1),
                                         tta_hflip=CFG().tta_hflip)
    _b1 = evaluate.binary_report(stages.stage1_scores(_lg1),
                                 stages.binary_targets(_y1),
                                 target_recall=CFG().target_recall_stage1)
    THR1 = float(_b1["threshold"])
    print(f"✅ 임계값 재계산: {THR1:.4f}  "
          f"(AUROC {_b1['auroc']:.4f} / precision {_b1['precision_at_target']:.3f})")
    (W/"stage1_threshold.json").write_text(json.dumps({
        "threshold": THR1, "auroc": _b1["auroc"],
        "precision_at_target": _b1["precision_at_target"],
        "target_recall": CFG().target_recall_stage1,
        "stage1_crop": STAGE1_CROP, "stage2_crop": BEST_CROP,
        "stage1_exp": name1, "stage2_exp": name2, "img_size": IMG_SIZE,
        "recovered_by": "notebook 05 (03 의 JSON 이 인계되지 않아 재계산)",
    }, indent=2, ensure_ascii=False))

## 1. 2단계 검증셋 성능 (보정 기준을 잡기 위해)

In [ ]:
dl_va2, ds_va2 = data.eval_loader(va2, cfg, model=m2, classes=CLASSES)
_, logits_va, y_va = train.evaluate_loader(m2, dl_va2, None, DEV, len(CLASSES), tta_hflip=True)
rep_va = evaluate.full_report(logits_va, y_va, CLASSES)
rep_va.plot_confusion()

## 2. 확률 보정 ★

신경망은 **자기 확신이 과합니다.** "95% 확신" 이라고 한 예측 100건 중
실제로는 70건만 맞는 게 흔합니다.

우리는 보호자에게 "신뢰도 72%" 같은 숫자를 보여줄 건데, 그게 거짓말이면 안 되죠.

**온도 스케일링**: logits 를 T 로 나누기만 합니다. 파라미터 딱 1개.
예측 순위는 전혀 안 바뀌니 **정확도는 그대로**, 확률만 정직해집니다.

⚠️ T 는 **검증셋**으로 학습하고 **holdout** 에서 효과를 확인합니다.
holdout 으로 T 를 맞추면 그것도 과적합입니다.

📖 [`docs/basics/08_확률보정과_임계값_결정.md`](../docs/basics/08_확률보정과_임계값_결정.md)

In [ ]:
T = calibrate.fit_temperature(logits_va, y_va)
# ⚠️ 03 에서 바로 온 경우 폴더 이름이 s2_{모델} 이 아닙니다 — 실제 찾은 경로에 씁니다.
(ck2.parent/"temperature.json").write_text(json.dumps({"temperature": T}, indent=2))
train.sync_to_persist(name2, files=("temperature.json",))   # 세션 밖에도 남깁니다

## 3. Holdout 최종 평가 — 파이프라인 기준 ★★

⚠️ **여기서부터는 되돌릴 수 없습니다.**
holdout 결과를 보고 하이퍼파라미터를 고치면 더 이상 holdout 이 아닙니다.
모든 결정이 끝난 뒤 딱 한 번만 여세요.

두 모델을 holdout 전체(정상 포함)에 **같은 순서로** 돌려 이어붙입니다.

In [ ]:
# 각 모델에는 그 모델이 학습한 크롭을 먹입니다
ho1_s2 = crop.switch_tag(ho1, BEST_CROP, verbose=False) if STAGE1_CROP != BEST_CROP else ho1
dl_h1, ds_h1 = data.eval_loader(ho1, cfg, model=m1, classes=CLASSES_STAGE1)
dl_h2, ds_h2 = data.eval_loader(ho1_s2, cfg, model=m2, classes=CLASSES)
assert len(ds_h1.df) == len(ds_h2.df)
assert (ds_h1.df["image_name"].to_numpy() == ds_h2.df["image_name"].to_numpy()).all()

_, lg_h1, _ = train.evaluate_loader(m1, dl_h1, None, DEV, len(CLASSES_STAGE1), tta_hflip=True)
_, lg_h2, _ = train.evaluate_loader(m2, dl_h2, None, DEV, len(CLASSES), tta_hflip=True)

s1_ho = stages.stage1_scores(lg_h1)
y_ho_final = ds_h1.df["label_orig"].to_numpy()

pipe_ho = stages.pipeline_report(s1_ho, lg_h2, y_ho_final, threshold=THR1)
stages.plot_pipeline_confusion(pipe_ho)

In [ ]:
# 2단계만 따로 본 holdout 점수 (파이프라인과 비교하기 위해)
mask_lesion = y_ho_final != NORMAL_LABEL
y_ho2 = torch.tensor([CLASSES.index(c) for c in y_ho_final[mask_lesion]])
rep_ho2 = evaluate.full_report(lg_h2[torch.as_tensor(mask_lesion)], y_ho2, CLASSES)
print(f"\n2단계만: macro-F1 {rep_ho2.metrics['macro_f1']:.4f}")
print(f"파이프라인: 최종 macro-F1 {pipe_ho['final_macro_f1']:.4f}, "
      f"스크리닝 recall {pipe_ho['lesion_screening_recall']:.4f}")
print("💡 두 숫자의 격차가 '1단계가 깎아먹는 양' 입니다. 보고할 때는 파이프라인 숫자를 쓰세요.")

In [ ]:
cal = calibrate.report(logits_va, y_va, lg_h2[torch.as_tensor(mask_lesion)], y_ho2)
probs_after = calibrate.apply(lg_h2[torch.as_tensor(mask_lesion)], T)
calibrate.reliability_diagram(evaluate.softmax_np(lg_h2[torch.as_tensor(mask_lesion)]),
                              probs_after, y_ho2.numpy())

## 4. Grad-CAM — 필수 검증 게이트 ★★

**정확도가 아무리 좋아도 여기서 통과 못 하면 그 모델은 실패입니다.**

이 데이터는 병변이 이미지의 5% 미만이고 배경이 제각각입니다.
모델이 병변이 아니라 진료대 무늬, 조명, 털 색을 보고 맞힐 수 있고,
그 단서가 클래스와 상관이 있으면 **검증 점수까지 잘 나옵니다.**

숫자로는 절대 못 잡습니다. 그림을 봐야 합니다.

In [ ]:
explain.grid(m2, va2, cfg, n=8)

In [ ]:
# 틀린 예측만 골라 보기 — 어디를 보고 틀렸는지가 개선의 힌트
va2b = va2.copy()
va2b["pred"] = [CLASSES[i] for i in evaluate.softmax_np(logits_va).argmax(1)]
explain.grid(m2, va2b, cfg, n=8, only_correct=False)

In [ ]:
# 수치화: CAM 이 실제 병변 박스와 얼마나 겹치는가
# 원본 이미지가 없는 환경(크롭만 업로드)이면 자동으로 크롭 좌표계로 계산합니다
overlap = explain.lesion_overlap_score(m2, va2, cfg, n=150, frame="auto")

### 🚦 게이트 판정

- `median_lift` ≥ 1.3 → 병변을 보고 있음, 통과
- `median_lift` < 1.3 → **배경 학습 의심**. 정확도와 무관하게 재작업

재작업 방향: 크롭 margin 축소 / 배경 증강 강화 / 세그멘테이션 마스킹

In [ ]:
if not overlap:
    print("⚠️ 정렬도를 계산하지 못했습니다 — 게이트를 판정할 수 없습니다.")
    print("   bbox 가 있는 행이 있는지, crop_path/img_w/img_h 가 살아있는지 확인하세요.")
    print("   이 상태로 배포 판단을 하면 안 됩니다.")
else:
    assert overlap["median_lift"] >= 1.3, (
        f"CAM-병변 정렬도 {overlap['median_lift']:.2f} < 1.3 — 배경을 보고 있을 가능성이 큽니다.\n"
        "정확도와 무관하게 재작업 대상입니다. docs/cautions/03 참고."
    )
    print(f"✅ Grad-CAM 게이트 통과 — median_lift {overlap['median_lift']:.2f} "
          f"(프레임: {overlap['frame']})")

### 🚦 실사용 견고성 게이트 ★

Grad-CAM 이 "병변을 보고 있다"고 해도, 그게 **어떤 배율에서만** 통하는 것일 수 있습니다.
보호자 사진의 배율과 병변 위치는 우리가 통제할 수 없습니다.

holdout 을 일부러 그렇게 망가뜨려 하락폭을 잽니다. 배포 판단의 마지막 관문입니다.

In [ ]:
from src import robust

rb = robust.report(m2, ho2, cfg, CLASSES, n=2000)
scale_drop = rb["scale"].get("_summary", {}).get("rel_drop", float("nan"))
shift_drop = rb["shift"].get("_summary", {}).get("rel_drop", float("nan"))

In [ ]:
if scale_drop == scale_drop and scale_drop > 0.30:
    print(f"🚨 배율 하락 {scale_drop:.0%} — 배포하면 안 됩니다.")
    print("   모델이 크롭 배율에 의존하고 있습니다. 보호자 사진에는 그 배율이 없습니다.")
    print("   → 고정 픽셀 크롭(f320) 또는 강한 배율 증강으로 다시 학습하세요 (노트북 03).")
elif scale_drop == scale_drop and scale_drop > 0.15:
    print(f"⚠️ 배율 하락 {scale_drop:.0%} — 실사용 성능은 holdout 점수보다 낮을 겁니다.")
    print("   배포한다면 그 사실을 문서에 명시하세요.")
else:
    print(f"✅ 배율 하락 {scale_drop:.0%}")

if shift_drop == shift_drop and shift_drop > 0.30:
    print(f"🚨 위치 하락 {shift_drop:.0%} — 병변이 화면 가운데 있을 때만 동작합니다.")
    print("   → 사용자에게 '병변을 가운데 두고 찍어주세요' 를 안내하거나, 위치 증강을 넣으세요.")
else:
    print(f"✅ 위치 하락 {shift_drop:.0%}")

## 5. 임계값과 거절(abstention) 설계

두 가지 임계값이 있습니다. 헷갈리기 쉬우니 구분하세요:

| 임계값 | 무엇을 정하나 | 기준 |
|---|---|---|
| **1단계 임계값** (`THR1`) | 병원에 가보라고 알릴지 | 재현율 ≥ 0.95 — 놓치지 않기 |
| **거절 임계값** (`abstain`) | 병변 **종류**를 말할지 | 틀릴 위험 ≤ 20% |

거절되면 종류를 말하지 않고 "판단이 어려운 사진입니다" 로 물러섭니다.
단, **1단계에서 이상이라고 판단했으면 거절해도 병원 안내는 유지**합니다 —
종류를 모른다는 게 괜찮다는 뜻은 아니니까요.

In [ ]:
cr = calibrate.coverage_risk_curve(probs_after, y_ho2.numpy())
thr_abstain = calibrate.suggest_abstain_threshold(probs_after, y_ho2.numpy(), max_risk=0.20)

### 1단계 임계값을 holdout 에서 재확인

⚠️ 임계값은 **검증셋에서 정하고** holdout 에서는 확인만 합니다.
holdout 점수가 목표(0.95)에 못 미치면 임계값을 고치는 게 아니라
"검증셋 기준으로 정한 임계값이 새 데이터에서는 recall 0.93 이었다"고 **보고**합니다.

In [ ]:
ybin_ho = (y_ho_final != NORMAL_LABEL).astype(int)
print(f"검증셋에서 정한 임계값 {THR1:.4f} 를 holdout 에 적용:")
print(f"  스크리닝 recall  {pipe_ho['lesion_screening_recall']:.4f}   (목표 ≥ 0.95)")
print(f"  헛알림 비율      {pipe_ho['false_alarm_rate']:.4f}")

# 참고용: holdout 에서 0.95를 만족하려면 어디였어야 하는가 (보고용, 채택하지 마세요)
ref = evaluate.binary_report(s1_ho, ybin_ho, target_recall=0.95)
print(f"\n(참고) holdout 기준 최적 임계값은 {ref['threshold']:.4f} 였습니다 — "
      "이 값을 채택하면 holdout 이 오염됩니다.")
print(f"두 값의 차이 {abs(ref['threshold'] - THR1):.4f} 가 크면 검증셋이 작거나 분포가 다른 것입니다.")

# ── 부위별로 쪼개서 봅니다 ★ (2026-08-26 추가) ──────────────────
# 왜 — 헛알림이 부위에 따라 **2.5배** 다릅니다 (다리 41.0% ↔ 연접부 16.6%).
#      "헛알림 33%" 하나로만 말하면 앱 팀이 어디가 약한지 모릅니다.
#      근거: docs/results/헛알림_사진통계_실측.md
REGION_KO = {"L": "다리", "H": "머리", "B": "몸통", "A": "연접부"}

# ⚠️ `ds_h1.df` 를 씁니다 — `y_ho_final` 과 `s1_ho` 가 여기서 나오므로
#    **순서가 보장**됩니다. `ho1` 을 쓰면 로더가 정렬을 바꿨을 때 조용히 어긋납니다.
_ho = ds_h1.df if "ds_h1" in dir() else None
if _ho is None or "region" not in _ho.columns:
    print("\n(부위 컬럼이 없어 부위별 보고를 건너뜁니다)")
else:
    import numpy as np
    import pandas as pd

    _ho = _ho.reset_index(drop=True)
    if not (len(_ho) == len(s1_ho) == len(y_ho_final)):
        raise SystemExit(f"❌ 길이가 다릅니다: df {len(_ho)} / 점수 {len(s1_ho)} / "
                         f"정답 {len(y_ho_final)} — 순서가 어긋났습니다.")
    _said = pd.Series(s1_ho >= THR1, index=_ho.index)
    _norm = pd.Series(y_ho_final == NORMAL_LABEL, index=_ho.index)
    g = pd.DataFrame({
        "정상": _norm.groupby(_ho["region"]).sum(),
        "헛알림": (_norm & _said).groupby(_ho["region"]).sum(),
        "병변": (~_norm).groupby(_ho["region"]).sum(),
        "놓침": ((~_norm) & (~_said)).groupby(_ho["region"]).sum(),
    })
    g["헛알림률"] = g["헛알림"] / g["정상"].replace(0, np.nan)
    g["놓침률"] = g["놓침"] / g["병변"].replace(0, np.nan)
    g = g[g["정상"] + g["병변"] >= 30].sort_values("헛알림률", ascending=False)

    print("\n" + "=" * 66)
    print(" 부위별 — 어디가 약한가 (holdout, 30장 이상)")
    print("=" * 66)
    print(f"  {'부위':<10}{'정상':>7}{'헛알림':>8}{'비율':>8}"
          f"{'병변':>7}{'놓침':>7}{'비율':>8}")
    print("  " + "─" * 56)
    for k, r in g.iterrows():
        lab = f"{k} ({REGION_KO[k]})" if str(k) in REGION_KO else str(k)
        print(f"  {lab[:9]:<10}{int(r['정상']):>7,}{int(r['헛알림']):>8,}"
              f"{r['헛알림률']:>8.1%}{int(r['병변']):>7,}{int(r['놓침']):>7,}"
              f"{r['놓침률']:>8.1%}")
    print("  " + "─" * 56)
    print(f"  {'전체':<10}{int(g['정상'].sum()):>7,}{int(g['헛알림'].sum()):>8,}"
          f"{g['헛알림'].sum() / max(g['정상'].sum(), 1):>8.1%}"
          f"{int(g['병변'].sum()):>7,}{int(g['놓침'].sum()):>7,}"
          f"{g['놓침'].sum() / max(g['병변'].sum(), 1):>8.1%}")
    _sp = g["헛알림률"].max() / max(g["헛알림률"].min(), 1e-9)
    print(f"\n  최고 ÷ 최저 = {_sp:.1f}배")
    print("  → 앱 팀에 전할 것: 사용자가 주로 어디를 찍느냐에 따라 체감이 다릅니다.")
    print("     털이 덮인 부위(다리·몸통)에서 헛알림이 많습니다 →")
    print("     촬영 문구 '털을 헤쳐서 피부가 보이게' 의 근거입니다.")
    REGION_TABLE = g.to_dict("index")


---
## 5-b. 촬영 가이드 (capture guideline) 도출 ★

배율 강건성(scale robustness)은 **모델링으로 못 잡았습니다.** 해상도로 8~9%p 를
줄인 뒤(224→384), 증강(augmentation) 7종을 한 실행에서 비교했지만 전부 잡음 안이었고
`zoom_both` 는 오히려 4%p 악화시켰습니다.
→ [`docs/results/STEP4B_증강스윕_실측.md`](../docs/results/STEP4B_증강스윕_실측.md)

남은 길은 **애초에 나쁜 배율이 안 들어오게 입력을 제한**하는 것입니다
(멘토 피드백 2번: "정확도가 가장 높은 scale 로 촬영하도록 가이드").

그러려면 "얼마나 가까이" 를 **숫자로** 말할 수 있어야 합니다. 그 숫자를 여기서 뽑습니다.

### 학습은 안 합니다

이미 학습된 모델에 **배율만 바꿔 추론**할 뿐입니다. 몇 분이면 끝납니다.

### 왜 격자를 촘촘하게 하나

지금까지 쓰던 5개 점(0.5 / 0.71 / 1 / 1.41 / 2)은 간격이 √2 라 너무 성깁니다.
03 실측으로 계산해보면 **밴드가 한 점으로 무너집니다**:

| 허용 하락 | 5개 점으로 계산한 밴드 |
|---|---|
| 5% 이내 | 1.0x ~ 1.0x ← 쓸 수 없음 |
| 10% 이내 | 1.0x ~ 1.41x |

1.41x 가 −5.2%, 0.71x 가 −10.4% 로 **둘 다 기준을 아슬하게 놓치기** 때문입니다.
0.85x / 1.2x 를 재봐야 실제 경계가 나옵니다.

### 배율을 "화면 점유율" 로 바꿉니다

보호자는 "1.2배" 를 모르지만 **"화면 절반"** 은 압니다.
학습 크롭이 `m1.5` 니 1x 에서 병변이 화면 가로의 1/1.5 = **67%** 를 차지합니다.


In [ ]:
# 배율 축 — 촘촘한 격자로 다시 잽니다 (추론만, 학습 없음)
guide_scale = robust.usable_range(
    m2, va2, cfg, CLASSES,
    zooms=(0.5, 0.6, 0.7, 0.85, 1.0, 1.2, 1.4, 1.7, 2.0),
    tolerances=(0.05, 0.10),
    crop_margin=CROP_MARGIN,  # 학습 크롭에서 유도 (m1.5→67% / m2.5→40%)
    n=3000, device=DEV)

# 위치 축 — 중앙에서 얼마나 벗어나도 되는지
guide_shift = robust.usable_shift(
    m2, va2, cfg, CLASSES,
    fracs=(0.0, 0.05, 0.10, 0.15, 0.20, 0.30),
    tolerance=0.05, n=3000, device=DEV)


In [ ]:
# ★ 보호자에게 보여줄 문구로 바꿉니다
b5 = guide_scale["bands"].get(0.05)
b10 = guide_scale["bands"].get(0.10)
occ = guide_scale["occupancy"]
mx = guide_shift.get("max_shift")

print("=" * 62)
print(" 촬영 가이드 (이 블록을 앱 UI 문구로 쓰세요)")
print("=" * 62)
if b5:
    print(f"  권장  : 병변이 화면 가로의 {occ[b5[0]]:.0%} ~ {occ[b5[1]]:.0%} 를 채우도록")
    print(f"          (배율 {b5[0]}x ~ {b5[1]}x · 성능 하락 5% 이내)")
if b10:
    print(f"  허용  : {occ[b10[0]]:.0%} ~ {occ[b10[1]]:.0%}  "
          f"(배율 {b10[0]}x ~ {b10[1]}x · 하락 10% 이내)")
if mx is not None:
    print(f"  위치  : 병변이 화면 중앙에서 {mx:.0%} 이내에 있도록")
print()
print("  → 촬영 UI 에 가이드 프레임을 띄우고, 벗어나면 셔터를 막거나")
print("     '조금 더 가까이 찍어주세요' 를 띄우는 방식으로 구현합니다.")
print("  → 이 구간을 벗어난 사진은 추론 대신 **다시 찍어달라고** 하는 게 맞습니다")
print("     (5번의 거절(abstention) 임계값과 같은 목적).")
print("=" * 62)

# 리포트에 남깁니다
import json
_W = env.work_root(); (_W/"reports").mkdir(parents=True, exist_ok=True)
(_W/"reports"/"capture_guide.json").write_text(json.dumps({
    "scale": {"peak": guide_scale["peak"], "table": guide_scale["table"],
              "bands": {str(k): v for k, v in guide_scale["bands"].items()},
              "occupancy": {str(k): v for k, v in guide_scale["occupancy"].items()},
              "crop_margin": guide_scale["crop_margin"]},
    "shift": {"max_shift": mx, "table": guide_shift["table"]},
}, indent=2, ensure_ascii=False))
print(f"저장: {_W/'reports'/'capture_guide.json'}")


## 6. 실제 사용 시뮬레이션

사용자가 사진 한 장을 올렸을 때 무엇이 보이는지 확인합니다.
`TwoStageEngine` 이 1단계 → 2단계를 실제 서비스와 같은 순서로 돌립니다.

⚠️ **병변 이름은 나오지 않습니다.** 2026-08-26 에 출력 형식을 바꿨습니다 —
1단계가 '이상' 이라고 하면 2단계 확률을 **여섯 개 전부** 보여주고 어느 쪽인지는
판단할 수 없다고 말한 뒤 진료를 권합니다. holdout 에서 그 이름이 **56.6%**
틀렸기 때문입니다 (`docs/cautions/03_의료AI_안전설계_원칙.md` §7-B).

In [ ]:
cfg_s2 = CFG(**{**cfg.to_dict(), "abstain_threshold": thr_abstain})
# 1단계 온도를 1.0 으로 박아두면 앞에서 보정한 게 무효가 됩니다.
# 체크포인트 옆의 temperature.json 을 읽고, 없으면 1.0 으로 물러섭니다.
_t1 = ck1.parent / "temperature.json"
T1_SERVE = json.loads(_t1.read_text(encoding="utf-8"))["temperature"] if _t1.exists() else 1.0
if not _t1.exists():
    print("⚠️ 1단계 temperature.json 이 없습니다 — 보정 안 된 확률로 시뮬레이션합니다.")
eng1 = infer.Engine(m1, cfg, CLASSES_STAGE1, temperature=T1_SERVE)
eng2 = infer.Engine(m2, cfg_s2, CLASSES, temperature=T)
pipeline = infer.TwoStageEngine(eng1, eng2, threshold=THR1)

# 정상 1장 + 병변 2장을 골라 봅니다
sample = pd.concat([
    ho1[ho1["label_orig"] == NORMAL_LABEL].sample(1, random_state=0),
    ho1[ho1["label_orig"] != NORMAL_LABEL].sample(2, random_state=0),
]) if (ho1["label_orig"] == NORMAL_LABEL).any() else ho1.sample(3, random_state=0)

for _, r in sample.iterrows():
    print("=" * 62)
    print("정답:", r["label_orig"], CLASS_KO.get(r["label_orig"], ""))
    pipeline.show(r["crop_path"])
    print()

## 7. 최종 리포트 저장

In [ ]:
summary = {
    # 부위별 성능 — 앱 팀이 "어디가 약한가" 를 알아야 합니다
    "by_region": (REGION_TABLE if "REGION_TABLE" in dir() else None),
    "stage1_hair_alpha": (STAGE1_HAIR_ALPHA if "STAGE1_HAIR_ALPHA" in dir() else 0.0),
    "model": BEST_MODEL, "stage2_crop": BEST_CROP, "stage1_crop": STAGE1_CROP,
    "stage1": {"threshold": THR1,
               "holdout_screening_recall": pipe_ho["lesion_screening_recall"],
               "holdout_false_alarm_rate": pipe_ho["false_alarm_rate"],
               "holdout_lesion_missed": pipe_ho["lesion_missed"]},
    "stage2_only": {"val_macro_f1": rep_va.metrics["macro_f1"],
                    "val_ci": list(rep_va.ci[1:]),
                    "holdout_macro_f1": rep_ho2.metrics["macro_f1"],
                    "holdout_ci": list(rep_ho2.ci[1:]),
                    "holdout_per_class_recall": rep_ho2.metrics["per_class"]["recall"]},
    "pipeline_holdout": {k: v for k, v in pipe_ho.items()
                         if k not in ("confusion", "per_class")},
    "pipeline_per_class": pipe_ho["per_class"],
    "calibration": cal,
    "temperature": T,
    "cam_lesion_overlap": overlap,
    "robustness": {"scale_rel_drop": scale_drop, "shift_rel_drop": shift_drop},
    "abstain_threshold": thr_abstain,
    "coverage_risk": cr,
}
p = W/"reports"/f"final_{BEST_MODEL}.json"
p.parent.mkdir(parents=True, exist_ok=True)
p.write_text(json.dumps(summary, indent=2, ensure_ascii=False))
print("저장:", p)
print(json.dumps(summary["pipeline_holdout"], indent=2, ensure_ascii=False))

---
## ✅ 배포 전 체크리스트

**파이프라인 기준**으로 확인하세요. 2단계만 좋아도 소용없습니다.

- [ ] **스크리닝 recall ≥ 0.95** (holdout) ← 가장 중요
- [ ] holdout 성능이 검증셋과 크게 다르지 않다 (차이 크면 과적합)
- [ ] 모든 병변 클래스의 recall 이 0.5 이상이다
- [ ] A5·A6(위험 병변)의 recall 이 특히 낮지 않다
- [ ] 보정 후 ECE < 0.10
- [ ] Grad-CAM 이 병변을 보고 있다 (median_lift ≥ 1.3)
- [ ] **배율 교란 하락 < 15%** — 보호자가 다른 거리에서 찍어도 버팀
- [ ] **위치 교란 하락 < 15%** — 병변이 정중앙이 아니어도 버팀
- [ ] 거절 임계값이 정해져 있다
- [ ] 모든 출력에 "진단이 아님" 문구가 붙는다
- [ ] **크롭 전제를 확인했다** — `m1.5`/`m2.5` 로 학습했다면 실제 사용자 사진에는
      병변 박스가 없습니다. `full` 점수를 기대치로 쓰거나 검출 모델을 앞에 붙이세요.

📖 반드시 읽기:
- [`docs/cautions/03_의료AI_안전설계_원칙.md`](../docs/cautions/03_의료AI_안전설계_원칙.md)
- [`docs/cautions/08_2단계_파이프라인_설계_주의점.md`](../docs/cautions/08_2단계_파이프라인_설계_주의점.md)